#### Используя Matlab или Python постройте (графики) функции импульсного отклика для всех эндогенных переменных в условиях шока технологического прогресса. 

#### Опишите полученные результаты интуитивно, сравнив IRF для случая 𝜑 = 0 и 𝜑 > 0.


Импульсная отклик (Impulse Response Function) – это динамика эндогенных переменных после неожиданного шока при условии, что до шока экономика находилась в стационарном состоянии. Проще говоря, IRF отвечает на вопрос: «Что происходит со всеми переменными экономики после единичного возмущающего воздействия (шока) на одну из экзогенных величин?»

#### Для построения IRF выполняются следующие шаги:

1. Поиск стационарного состояния: Сначала необходимо вычислить стационарное (устойчивое) состояние модели – значения всех переменных в отсутствии шоков. Это включает решение уравнений равновесия в стационаре (например, уравнений Эйлера, баланса ресурсов, накопления капитала и т.д.) для постоянных значений переменных.

2. Линеаризация модели: Далее модель линеаризуется (обычно лог-линеаризуется) вокруг стационарного состояния. Это значит, что все уравнения приводятся к приближенно линейной форме по отклонениям переменных от стационарного уровня

3. Инициализация шока: Предполагается, что до воздействия (на t = 0) экономика находилась точно в стационарном состоянии (все отклонения равны нулю). Затем в момент t = 1 накладывается разовый шок – например, технологическая переменная A_t увеличивается на 1% (в логарифмическом выражении) или на одну условную единицу. Этот шок задаётся как вектор ε_1, в котором соответствующий компонент равен единице, а все прочие – нули. Важная деталь: переменные-состояния не могут прыгнуть в момент шока. Например, капитал в периоде t = 1 ещё не успевает измениться, так что первоначально весь эффект шока отражается на контрольных переменных (таких как выпуск, потребление, инвестиции). Потребление и инвестиции могут сразу среагировать, а капитал начнёт меняться только со следующего периода за счёт дополнительного инвестирования.

4. Расчёт траектории отклика: Используя линеаризованную систему или напрямую решая модель, вычисляется траектория отклонений всех эндогенных переменных в последующие периоды после шока. В линейной модели это сводится к итеративному применению матрицы перехода Φ и влияния шока Ψ. Проще говоря, в момент t = 1 мы фиксируем ε_1, (например, шок технологического прогресса), рассчитываем изменения переменных, затем пусть ε_t=0 для t > 1, и прослеживаем, как система возвращается к стационару. Импульсные отклики – это графики, показывающие, как, например, выпуск, потребление, капитал и пр. отклоняются от стационарного уровня в каждый период после шока, пока постепенно не вернутся к равновесию.

In [1]:
# Задание параметров модели (пример)
parameters = {
    'alpha': 0.35,    # доля капитала в выпуске
    'beta': 0.99,     # коэффициент межвременных предпочтений
    'delta': 0.025,   # норма выбытия капитала (ежеквартальная)
    'rhoa': 0.90,     # устойчивость (автокорреляция) технологического шока
    'sigma': 1.5      # коэффициент CRRA (обратная эластичность по потреблению)
}
# Определение уравнений модели: уравнение Эйлера, ресурсное ограничение, процесс технологии
def equilibrium_equations(fwd, cur, p):
    euler = p['beta'] * (fwd['c']**(-p['sigma'])) * (p['alpha']*fwd['a']*fwd['k']**(p['alpha']-1) + 1 - p['delta']) \
            - cur['c']**(-p['sigma'])
    resource = cur['c'] + fwd['k'] - (1-p['delta'])*cur['k'] - cur['a']*(cur['k']**p['alpha'])
    tech_process = p['rhoa'] * np.log(cur['a']) - np.log(fwd['a'])  # AR(1) for technology in logs
    return np.array([euler, resource, tech_process])

# Решение модели
model = ls.model(equations=equilibrium_equations, n_states=2, n_exo_states=1,
                 variables=['a','k','c'], parameters=parameters)
model.compute_ss([1,1,1])           # вычисление стационарного состояния
model.approximate_and_solve()      # линеаризация и решение системы
irf = model.impulse(T=40, t0=5, shocks=[0.01])  # импульсный отклик на 1% шок технологии


NameError: name 'ls' is not defined

В этом примере задаётся стандартная RBC-модель: (1) уравнение Эйлера для потребления-инвестиций, (2) ограничение ресурса, (3) авторегрессия для технологического процесса A_t

Параметры взяты близкими к общепринятым калибровкам (см. ниже). После нахождения стационара и решения система IRF вычисляется вызовом model.impulse(...), где имитируется шок величиной 1% в A_t. Результат – массив откликов переменных (например, отклонения для a,k,c) по периодам, которые можно визуализировать графически.

#### Калибровка параметров модели

Калибровка модели – это подбор значений структурных параметров (таких как β, δ, α, и др.) на основе внешних эмпирических соображений или предыдущих исследований. В задании особо подчеркнуто, что выбор параметров должен быть обоснован ссылками на источники, иначе работа не получит баллов. Обычно при калибровке динамических моделей общего равновесия исследователи используют либо стилизованные факты экономики (средние доли, соотношения и темпы роста), либо результаты микроэкономических оценок, либо подгоняют параметры по определённым целевым статистикам (методCalibration).